# VitalDB ICU Stay Duration

Same setup as the admission notebook but the target is now 3 classes: no ICU, short stay (1-3 days), prolonged (4+ days). Went with this cutoff because something like 68% of all ICU stays in this data are exactly 1 day, which reads more like routine post-op monitoring than someone actually being sick. 4+ days felt like a reasonable line for "this patient is genuinely in trouble."

Same holdout-first, SMOTE-in-folds approach as before. Metrics are one-vs-rest macro averaged across the three classes so the rare prolonged-stay group doesn't get washed out by the no-ICU majority.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import json
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

In [ ]:
df = pd.read_csv('clinical_data.csv')
vitals_df = pd.read_csv('vitaldb_hr_spo2.csv')
ecg_df = pd.read_csv('vitaldb_ecg_features.csv')
ppg_df = pd.read_csv('vitaldb_ppg_features.csv')

data = df.copy()
data['ecg_abnormal'] = (data['preop_ecg'] != 'Normal Sinus Rhythm').astype(int)
for c in ['sex', 'department', 'optype']:
    if c in data.columns:
        data[c] = LabelEncoder().fit_transform(data[c].astype(str))

def bucket_icu(days):
    if days <= 0:
        return 0
    elif days <= 3:
        return 1
    return 2

data['icu_class'] = data['icu_days'].apply(bucket_icu)
print(data['icu_class'].value_counts().sort_index())

clinical_feats = [
    'age', 'sex', 'bmi', 'asa', 'emop',
    'preop_htn', 'preop_dm', 'ecg_abnormal',
    'preop_hb', 'preop_plt', 'preop_pt', 'preop_aptt',
    'preop_na', 'preop_k', 'preop_gluc', 'preop_alb',
    'preop_cr', 'preop_bun', 'preop_ast', 'preop_alt'
]
vitals_feats = ['hr_mean', 'hr_min', 'hr_max', 'hr_std', 'spo2_mean', 'spo2_min']
ppg_feats = ['ppg_mean', 'ppg_median', 'ppg_std', 'ppg_iqr', 'ppg_range', 'ppg_skew', 'ppg_kurtosis', 'ppg_energy', 'ppg_mad', 'ppg_min', 'ppg_max']
ecg_feats = ['ecg_mean', 'ecg_median', 'ecg_std', 'ecg_iqr', 'ecg_range', 'ecg_skew', 'ecg_kurtosis', 'ecg_energy', 'ecg_mad', 'ecg_min', 'ecg_max']

In [ ]:
def make_X(df, cols):
    X = df[cols].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median())


def macro_pr(y_true, proba):
    # one-vs-rest average precision, then averaged across the 3 classes
    scores = []
    for c in range(proba.shape[1]):
        y_bin = (y_true == c).astype(int)
        scores.append(average_precision_score(y_bin, proba[:, c]))
    return np.mean(scores)


def fit_with_cv(X, y, name):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros((len(y), 3))
    aucs, prs = [], []
    for tr_i, val_i in kf.split(X, y):
        Xtr, ytr = X.iloc[tr_i], y.iloc[tr_i]
        Xval, yval = X.iloc[val_i], y.iloc[val_i]
        Xtr_s, ytr_s = SMOTE(random_state=SEED).fit_resample(Xtr, ytr)
        clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, objective='multi:softprob', num_class=3, random_state=SEED, verbosity=0)
        clf.fit(Xtr_s, ytr_s)
        proba = clf.predict_proba(Xval)
        oof[val_i] = proba
        aucs.append(roc_auc_score(yval, proba, multi_class='ovr', average='macro'))
        prs.append(macro_pr(yval.values, proba))

    cv_auc = roc_auc_score(y, oof, multi_class='ovr', average='macro')
    cv_pr = macro_pr(y.values, oof)
    print(f"{name}: cv macro auc {cv_auc:.4f} (+/- {np.std(aucs):.4f}), cv macro pr {cv_pr:.4f} (+/- {np.std(prs):.4f})")

    Xs, ys = SMOTE(random_state=SEED).fit_resample(X, y)
    final_clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, objective='multi:softprob', num_class=3, random_state=SEED, verbosity=0)
    final_clf.fit(Xs, ys)
    return final_clf, cv_auc, cv_pr


def boot_ci(y_true, proba, fn, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 3:
            continue
        vals.append(fn(y_true[idx], proba[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def boot_delta(y_true, proba_a, proba_b, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 3:
            continue
        a = roc_auc_score(y_true[idx], proba_a[idx], multi_class='ovr', average='macro')
        b = roc_auc_score(y_true[idx], proba_b[idx], multi_class='ovr', average='macro')
        vals.append(b - a)
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)


def eval_on_holdout(clf, X_test, y_test, name):
    proba = clf.predict_proba(X_test)
    y_arr = np.array(y_test)
    auc_m, auc_lo, auc_hi = boot_ci(y_arr, proba, lambda y, p: roc_auc_score(y, p, multi_class='ovr', average='macro'))
    pr_m, pr_lo, pr_hi = boot_ci(y_arr, proba, macro_pr)
    print(f"{name} holdout -- macro auc {auc_m:.4f} ({auc_lo:.4f}-{auc_hi:.4f}), macro pr {pr_m:.4f} ({pr_lo:.4f}-{pr_hi:.4f})")
    return proba, {'macro_auc': auc_m, 'auc_ci': (auc_lo, auc_hi), 'macro_pr': pr_m, 'pr_ci': (pr_lo, pr_hi)}

## ECG side

In [ ]:
ecg_merged = data.merge(vitals_df, on='caseid').merge(ecg_df, on='caseid')
print(f"ecg cohort: {len(ecg_merged)}")
print(ecg_merged['icu_class'].value_counts().sort_index())

ecg_train, ecg_test = train_test_split(ecg_merged, test_size=0.2, stratify=ecg_merged['icu_class'], random_state=SEED)
ecg_train = ecg_train.reset_index(drop=True)
ecg_test = ecg_test.reset_index(drop=True)
y_tr_ecg = ecg_train['icu_class'].astype(int)
y_te_ecg = ecg_test['icu_class'].astype(int)

ecg_runs = {}
for name, feats in [
    ('ecg', ecg_feats),
    ('clinical', clinical_feats),
    ('vitals', vitals_feats),
    ('ecg_clinical', ecg_feats + clinical_feats),
    ('clinical_vitals', clinical_feats + vitals_feats),
    ('full', ecg_feats + clinical_feats + vitals_feats),
]:
    Xtr = make_X(ecg_train, feats)
    Xte = make_X(ecg_test, feats)
    clf, cv_auc, cv_pr = fit_with_cv(Xtr, y_tr_ecg, name)
    proba, metrics = eval_on_holdout(clf, Xte, y_te_ecg, name)
    metrics['cv_auc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    ecg_runs[name] = {'proba': proba, 'metrics': metrics}

## PPG side

In [ ]:
ppg_merged = data.merge(vitals_df, on='caseid').merge(ppg_df, on='caseid')
print(f"ppg cohort: {len(ppg_merged)}")
print(ppg_merged['icu_class'].value_counts().sort_index())

ppg_train, ppg_test = train_test_split(ppg_merged, test_size=0.2, stratify=ppg_merged['icu_class'], random_state=SEED)
ppg_train = ppg_train.reset_index(drop=True)
ppg_test = ppg_test.reset_index(drop=True)
y_tr_ppg = ppg_train['icu_class'].astype(int)
y_te_ppg = ppg_test['icu_class'].astype(int)

ppg_runs = {}
for name, feats in [
    ('ppg', ppg_feats),
    ('clinical', clinical_feats),
    ('vitals', vitals_feats),
    ('ppg_clinical', ppg_feats + clinical_feats),
    ('clinical_vitals', clinical_feats + vitals_feats),
    ('full', ppg_feats + clinical_feats + vitals_feats),
]:
    Xtr = make_X(ppg_train, feats)
    Xte = make_X(ppg_test, feats)
    clf, cv_auc, cv_pr = fit_with_cv(Xtr, y_tr_ppg, name)
    proba, metrics = eval_on_holdout(clf, Xte, y_te_ppg, name)
    metrics['cv_auc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    ppg_runs[name] = {'proba': proba, 'metrics': metrics}

In [ ]:
def show_deltas(runs, y_test, sig):
    y_arr = np.array(y_test)
    a = boot_delta(y_arr, runs['clinical']['proba'], runs[f'{sig}_clinical']['proba'])
    b = boot_delta(y_arr, runs[sig]['proba'], runs[f'{sig}_clinical']['proba'])
    c = boot_delta(y_arr, runs['clinical_vitals']['proba'], runs[f'{sig}_clinical']['proba'])
    d = boot_delta(y_arr, runs[f'{sig}_clinical']['proba'], runs['full']['proba'])
    e = boot_delta(y_arr, runs['clinical_vitals']['proba'], runs['full']['proba'])
    print(f"\n{sig}:")
    print(f"  signal+clin vs clin alone:      {a[0]:+.4f} ({a[1]:+.4f}, {a[2]:+.4f})")
    print(f"  signal+clin vs signal alone:    {b[0]:+.4f} ({b[1]:+.4f}, {b[2]:+.4f})")
    print(f"  signal+clin vs vitals+clin:     {c[0]:+.4f} ({c[1]:+.4f}, {c[2]:+.4f})")
    print(f"  full vs signal+clin:            {d[0]:+.4f} ({d[1]:+.4f}, {d[2]:+.4f})")
    print(f"  full vs clin+vitals:            {e[0]:+.4f} ({e[1]:+.4f}, {e[2]:+.4f})")

show_deltas(ecg_runs, y_te_ecg, 'ecg')
show_deltas(ppg_runs, y_te_ppg, 'ppg')

## Stacking ECG and PPG together, ordinal version

Same question as in the admission notebook, just on the stay-length outcome this time. Intersection cohort (both signals extracted), clinical+vitals baseline, then add ECG, PPG, and both on top.

In [ ]:
combo_merged = data.merge(vitals_df, on='caseid').merge(ecg_df, on='caseid').merge(ppg_df, on='caseid')
print(f"intersection cohort: {len(combo_merged)}")
print(combo_merged['icu_class'].value_counts().sort_index())

combo_train, combo_test = train_test_split(combo_merged, test_size=0.2, stratify=combo_merged['icu_class'], random_state=SEED)
combo_train = combo_train.reset_index(drop=True)
combo_test = combo_test.reset_index(drop=True)
y_tr_combo = combo_train['icu_class'].astype(int)
y_te_combo = combo_test['icu_class'].astype(int)

combo_runs = {}
for name, feats in [
    ('clinical_vitals', clinical_feats + vitals_feats),
    ('clinical_vitals_ecg', clinical_feats + vitals_feats + ecg_feats),
    ('clinical_vitals_ppg', clinical_feats + vitals_feats + ppg_feats),
    ('clinical_vitals_both', clinical_feats + vitals_feats + ppg_feats + ecg_feats),
]:
    Xtr = make_X(combo_train, feats)
    Xte = make_X(combo_test, feats)
    clf, cv_auc, cv_pr = fit_with_cv(Xtr, y_tr_combo, name)
    proba, metrics = eval_on_holdout(clf, Xte, y_te_combo, name)
    metrics['cv_auc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    combo_runs[name] = {'proba': proba, 'metrics': metrics}

In [ ]:
y_arr_combo = np.array(y_te_combo)
ppg_over_ecg = boot_delta(y_arr_combo, combo_runs['clinical_vitals_ecg']['proba'], combo_runs['clinical_vitals_ppg']['proba'])
both_over_ppg = boot_delta(y_arr_combo, combo_runs['clinical_vitals_ppg']['proba'], combo_runs['clinical_vitals_both']['proba'])

print(f"ppg vs ecg on top of clinical+vitals: {ppg_over_ecg[0]:+.4f} ({ppg_over_ecg[1]:+.4f}, {ppg_over_ecg[2]:+.4f})")
print(f"ecg added on top of ppg: {both_over_ppg[0]:+.4f} ({both_over_ppg[1]:+.4f}, {both_over_ppg[2]:+.4f})")

In [ ]:
out = {
    'ecg': {k: v['metrics'] for k, v in ecg_runs.items()},
    'ppg': {k: v['metrics'] for k, v in ppg_runs.items()},
    'combined': {k: v['metrics'] for k, v in combo_runs.items()},
}
with open('vitaldb_icu_stay_duration_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)